In [1]:
%matplotlib ipympl
import sys

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure

configure(level="DEBUG", stream=sys.stdout, fmt="text")

NodeRegistry with 0 registered nodes: []
Successfully registered nodes from module: tsut.components.nodes.data_sources._register
Successfully registered nodes from module: tsut.components.nodes.metrics.classification._register
Successfully registered nodes from module: tsut.components.nodes.metrics.regression._register
Successfully registered nodes from module: tsut.components.nodes.models._register
Successfully registered nodes from module: tsut.components.nodes.sinks._register
Successfully registered nodes from module: tsut.components.nodes.transforms.encodings._register
Successfully registered nodes from module: tsut.components.nodes.transforms.feature_selection._register
Successfully registered nodes from module: tsut.components.nodes.transforms.filters._register
Successfully registered nodes from module: tsut.components.nodes.transforms.imputations._register
Successfully registered nodes from module: tsut.components.nodes.transforms.operations._register
Successfully registered nod

<Logger tsut (DEBUG)>

In [2]:
NODE_REGISTRY.list()

In [3]:
csv_config_class = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")
csv_running_config_class = NODE_REGISTRY["TabularCSVFetcher"]["running_config"]

In [4]:
csv_running_config_class

tsut.components.nodes.data_sources.tabular_csv_fetcher.TabularCSVFetcherRunningConfig

In [5]:
data_csv_config = csv_config_class()
target_csv_config = csv_config_class()

In [6]:
data_csv_config.running_config.csv_path = "../data/fake_batch.csv"
data_csv_config.running_config.context_path = "../data/fake_batch_context.json"

target_csv_config.running_config.csv_path = "../data/fake_target_df.csv"
target_csv_config.running_config.context_path = "../data/fake_target_context.json"

In [7]:
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

In [8]:
missing_filter_config_class = NODE_REGISTRY.get_node_config_class("MissingRateFilter")
data_missing_filter_config = missing_filter_config_class()
target_missing_filter_config = missing_filter_config_class()

target_missing_filter_config.in_ports["input"].mode=["training", "evaluation"]
target_missing_filter_config.hyperparameters.threshold = 0.0

In [9]:
nodes ={
        "data_source": ("TabularCSVFetcher", data_csv_config),
        "target_source": ("TabularCSVFetcher", target_csv_config),
        "data_missing_filter": ("MissingRateFilter", data_missing_filter_config),
        "category_splitter": ("DataCategoryFilter", NODE_REGISTRY.get_node_config_class("DataCategoryFilter")()),
        "one_hot_encoder": ("OneHotEncoding", NODE_REGISTRY.get_node_config_class("OneHotEncoding")()),
        "target_missing_filter": ("MissingRateFilter", target_missing_filter_config),
        "iqr_filter": ("IQROutlierFilter", NODE_REGISTRY.get_node_config_class("IQROutlierFilter")()),
        "variance_threshold_filter": ("VarianceFilter", NODE_REGISTRY.get_node_config_class("VarianceFilter")()),
        "standard_scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
        "feature_concatenate": ("FeatureConcatenate", NODE_REGISTRY.get_node_config_class("FeatureConcatenate")()),
        "random_forest_regressor": ("RandomForestRegressor", NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()),
        "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
        "mse": ("MSE", NODE_REGISTRY.get_node_config_class("MSE")()),
        "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

In [10]:
edges = [
    Edge(source="data_source", target="data_missing_filter", ports_map=[("output", "input")]),
    Edge(source="data_missing_filter", target="category_splitter", ports_map=[("output", "input")]),
    Edge(source="one_hot_encoder", target="feature_concatenate", ports_map=[("output", "input_2")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("numerical", "input")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("categorical", "sliced")]),
    Edge(source="target_source", target="iqr_filter", ports_map=[("output", "target")]),
    Edge(source="iqr_filter", target="variance_threshold_filter", ports_map=[("output", "input")]),
    Edge(source="iqr_filter", target="one_hot_encoder", ports_map=[("sliced", "input")]),
    Edge(source="variance_threshold_filter", target="standard_scaler", ports_map=[("output", "input")]),
    Edge(source="standard_scaler", target="feature_concatenate", ports_map=[("output", "input_1")]),
    Edge(source="feature_concatenate", target="random_forest_regressor", ports_map=[("output", "X")]),
    Edge(source="iqr_filter", target="target_missing_filter", ports_map=[("target", "input")]),
    Edge(source="target_missing_filter", target="random_forest_regressor", ports_map=[("output", "y")]),
    Edge(source="random_forest_regressor", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="r2", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="mse", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="mse", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="sink", ports_map=[("pred", "dump")]),
]

In [11]:
pipe_conf = PipelineConfig(
    nodes=nodes,
    edges=edges,
)

pipe = Pipeline(config=pipe_conf)

09:12:11 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_source  target=iqr_filter  source_port=output  target_port=target
09:12:11 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=iqr_filter  target=one_hot_encoder  source_port=sliced  target_port=input
09:12:11 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=random_forest_regressor  source_port=output  target_port=y
09:12:11 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=r2  source_port=output  target_port=target
09:12:11 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=mse  source_port=output  target_port=target
09:12:11 WARNING  [tsut.pipeline] Data structure mismatch (compatible but not identical)  source=random_forest_regressor  target=sink  source_port=

In [12]:
pipe.render()

In [13]:
from tsut.core.pipeline.runners.smart_runner import SmartRunner

In [14]:

pipe.compile()
runnable = SmartRunner(pipe)

09:12:13 INFO     [tsut.pipeline] Phase compilation start  pipeline_name=My Pipeline  pipeline_phase=compilation  phase_status=start
09:12:13 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_source  target=iqr_filter  source_port=output  target_port=target
09:12:13 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=iqr_filter  target=one_hot_encoder  source_port=sliced  target_port=input
09:12:13 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=random_forest_regressor  source_port=output  target_port=y
09:12:13 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=r2  source_port=output  target_port=target
09:12:13 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=mse  source_port=output  target_port=target
09:12:13 WARNI

In [15]:
runnable.train()

09:12:13 INFO     [tsut.runner.smart] Phase training start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=training  phase_status=start
09:12:13 DEBUG    [tsut.runner.smart] Node called: sink  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=sink  pipeline_phase=training
09:12:13 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=training
09:12:13 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=training
09:12:13 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=training
09:12:13 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=training
09:

In [16]:
runnable.infer()

09:12:13 INFO     [tsut.runner.smart] Phase inference start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=inference  phase_status=start
09:12:13 DEBUG    [tsut.runner.smart] Node called: sink  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=sink  pipeline_phase=inference
09:12:13 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=inference
09:12:13 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=inference
09:12:13 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=inference
09:12:13 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=infere

{'dump': <tsut.core.common.data.data.TabularData at 0x125dadb70>}

In [17]:
eval_metrics = runnable.evaluate()

09:12:13 INFO     [tsut.runner.smart] Phase evaluation start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=evaluation  phase_status=start
09:12:13 DEBUG    [tsut.runner.smart] Node called: r2  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=r2  pipeline_phase=evaluation
09:12:13 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=evaluation
09:12:13 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=evaluation
09:12:13 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=evaluation
09:12:13 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=eval